In [1]:
import os, json
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import norm
from scipy.stats import zscore
from src import Questions
import matplotlib.patches as mpatches
from functools import reduce


plt.rcParams.update({'font.size': 17})

In [29]:
# creating a dataframe of all of the annotator profiles
EXCLUDED_SCORES = ["2_13", "23_", "24_", "25_", "27_"]
questions = Questions().get_questions()
SCORED_ANNOTATOR_DIR = "annotation_artefacts/final_annotator_objects"
os.makedirs(SCORED_ANNOTATOR_DIR, exist_ok=True)

# making a raw scores (both characteristic and downstream) for each annotator
all_tasktype_df_by_model = {}
all_models = [
    "gpt_4o",
    "llama31_70b_Inst",
    "llama31_8B_Inst",
    "llama32_1B_Inst",
    "qwen25_72b_Inst",
    "Qwen25_7B_Inst",
    "Qwen25_1dot5B_Inst"
]

color_dict = {
    "imperative" : "#1f77b4",
    "interrogative" : "#ff7f0e",
    "declarative" : "#2ca02c",
    "correct" : "#6084f7",
    "incorrect" : "#d6425d",
    "white" : "#ffffff",
}

task_types = []
for i, k in enumerate(questions.keys()):
    task_types.append(questions[k]["question_type"])
task_types = list(set(task_types))

for presented_model in all_models:
    all_tasktype_df = {}
    for tt in task_types:
        downstream_score_df = []
        for annfilename in os.listdir(SCORED_ANNOTATOR_DIR):
            annotator_filepath = os.path.join(SCORED_ANNOTATOR_DIR, annfilename)
            with open(annotator_filepath, "r") as  ao:
                annotator_object = json.load(ao)
            
            downstream_scores = {}
            for question_key, values in annotator_object["answers"].items():
                if values["answer"] == None or questions[question_key]["question_type"] != tt:
                    continue
                for qid, score_s in values["downstream_scores"][presented_model].items():
                    postfix = questions[question_key]["metrics"][0] # first metric entry is by convention default
                    matching_keys = [k for k in score_s if k.endswith(f"{postfix}_score")]
                    if not matching_keys:
                        raise ValueError(f"No key ends with '{postfix}' for qid {qid}")
                    correct_key = matching_keys[0]
                    downstream_scores[f"{qid}-sentence_type"] = values["sentence_type"]
                    downstream_scores[f"{qid}-sentence_length"] = values["sentence_length"]
                    downstream_scores[f"{qid}-sentence_grammaticality"] = values["grammaticality"]
                    downstream_scores[correct_key] = score_s[correct_key]

            downstream_score_df.append({
                "id" : annotator_object["id"],
                **downstream_scores,
            })
        all_tasktype_df[tt] = pd.DataFrame(downstream_score_df)
    all_tasktype_df_by_model[presented_model] = all_tasktype_df
    

In [30]:
print(all_tasktype_df_by_model.keys())
print(all_tasktype_df_by_model['gpt_4o'].keys())

dict_keys(['gpt_4o', 'llama31_70b_Inst', 'llama31_8B_Inst', 'llama32_1B_Inst', 'qwen25_72b_Inst', 'Qwen25_7B_Inst', 'Qwen25_1dot5B_Inst'])
dict_keys(['syntaxes', 'conditioned_text_gen', 'qa_extraction', 'semantics', 'machine_translation', 'knowledge_and_reasoning', 'summarization'])


In [28]:
from scipy.stats import f_oneway
from scipy.stats import ttest_ind
from itertools import combinations

def compute_anova_score(data_dict):
    # Extract the lists of scores
    groups = list(data_dict.values())
    # Perform one-way ANOVA
    f_statistic, p_value = f_oneway(*groups)
    return f_statistic, p_value

def pairwise_p_values_with_higher(data_dict):
    results = {}
    for (key1, values1), (key2, values2) in combinations(data_dict.items(), 2):
        _, p = ttest_ind(values1, values2, equal_var=False)
        
        mean1 = sum(values1) / len(values1)
        mean2 = sum(values2) / len(values2)
        higher = key1 if mean1 > mean2 else key2

        results[(key1, key2)] = {
            "p" : p,
            "higher" : higher,
        }
    return results

def df_with_hexcolor_to_latex(values_df, color_df):
    from collections import OrderedDict

    latex_rows = []
    defined_colors = OrderedDict()
    color_idx = 0

    for i in range(values_df.shape[0]):
        row = []
        for j in range(values_df.shape[1]):
            val = values_df.iat[i, j]
            hex_color = color_df.iat[i, j]
            if isinstance(hex_color, str) and hex_color.startswith("#"):
                if hex_color not in defined_colors:
                    defined_colors[hex_color] = f"customcolor{color_idx}"
                    color_idx += 1
                colorname = defined_colors[hex_color]
                cell = f"\\cellcolor{{{colorname}!30}}{val}"
            else:
                cell = str(val)
            row.append(cell)
        latex_rows.append(" & ".join(row) + r" \\")
    
    col_headers = " & ".join(values_df.columns) + r" \\"
    header = r"\begin{tabular}{" + "l" * values_df.shape[1] + "}\n\\hline\n" + col_headers + "\n\\hline"
    body = "\n".join(latex_rows)
    footer = "\n\\hline\n\\end{tabular}"

    color_defs = "\n".join(
        [f"\\definecolor{{{name}}}{{HTML}}{{{hex[1:]}}}" for hex, name in defined_colors.items()]
    )

    return color_defs + "\n\n" + header + "\n" + body + footer

In [64]:
# NAIVE P VALUES
REJECT_THRESHOLD = 0.05
# 0 for TONE, 2 for GRAMMAR
CHAR_COLUMN = 2
task_lingcat_scores_by_model = {}
for mmm in all_tasktype_df_by_model.keys():
    task_lingcat_scores_by_model[mmm] = {}
    for tt in task_types:
        task_lingcat_scores_by_model[mmm][tt] = {}
        taskdf = all_tasktype_df_by_model[mmm][tt].drop(columns=['id'])
        cols = list(taskdf.columns)
        for i in range(0, len(cols), 4):
            chunk = cols[i:i+4]
            task_qidx = chunk[-1].split("-")[-2]
            if not any(task_qidx.startswith(prefix) for prefix in EXCLUDED_SCORES):
                task_lingcat_scores_by_model[mmm][tt][task_qidx] = {}
                chunkdf = taskdf[chunk].dropna()
                for ptone, tone_group in chunkdf.groupby(chunk[CHAR_COLUMN]):
                    task_lingcat_scores_by_model[mmm][tt][task_qidx][ptone] = tone_group[chunk[-1]].to_list()

rejectable_objlist =[]
for modelname in task_lingcat_scores_by_model.keys():
    row_instance = {"model" : modelname}
    for tg in task_lingcat_scores_by_model[modelname].keys():
        unfair_tasks = []
        for qidx in task_lingcat_scores_by_model[modelname][tg].keys():
            this_task_unfair = False
            ttest_result = pairwise_p_values_with_higher(task_lingcat_scores_by_model[modelname][tg][qidx])
    
            for ttest_values in ttest_result.values():
                if ttest_values['p'] < REJECT_THRESHOLD:
                    this_task_unfair = True
            unfair_tasks.append(this_task_unfair)
        row_instance[tg] = sum(unfair_tasks)/len(unfair_tasks) 
    rejectable_objlist.append(row_instance)
rejectable_df = pd.DataFrame(rejectable_objlist)

if CHAR_COLUMN == 0:
    rejectable_df.to_csv(f"score_tables/ptone_ptt.csv")
else:
    rejectable_df.to_csv(f"score_tables/pgrammar_ptt.csv")

/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:573: RuntimeWarning: Precision loss occurred in moment calculation due to catastrophic cancellation. This occurs when the data are nearly identical. Results may be unreliable.
  res = hypotest_fun_out(*samples, **kwds)


In [62]:
rejectable_df

,model,syntaxes,conditioned_text_gen,qa_extraction,semantics,machine_translation,knowledge_and_reasoning,summarization
0,gpt_4o,0.00,0.0,0.000000,0.333333,0.857143,0.230769,0.0
1,llama31_70b_Inst,0.25,1.0,0.333333,0.333333,0.285714,0.153846,0.0
2,llama31_8B_Inst,0.75,1.0,0.000000,0.666667,0.714286,0.076923,0.0
3,llama32_1B_Inst,0.25,0.0,0.000000,0.000000,0.857143,0.230769,0.0
4,qwen25_72b_Inst,0.25,1.0,0.000000,0.333333,0.571429,0.384615,0.0
5,Qwen25_7B_Inst,0.75,1.0,0.000000,0.333333,0.285714,0.461538,0.0
6,Qwen25_1dot5B_Inst,0.00,0.0,0.666667,0.333333,0.714286,0.076923,0.0


In [42]:
print(task_lingcat_scores_by_model.keys())
print(task_lingcat_scores_by_model['gpt_4o'].keys())
print(task_lingcat_scores_by_model['gpt_4o']['syntaxes'].keys())
print(task_lingcat_scores_by_model['gpt_4o']['syntaxes']['16_0'].keys())

dict_keys(['gpt_4o', 'llama31_70b_Inst', 'llama31_8B_Inst', 'llama32_1B_Inst', 'qwen25_72b_Inst', 'Qwen25_7B_Inst', 'Qwen25_1dot5B_Inst'])
dict_keys(['syntaxes', 'conditioned_text_gen', 'qa_extraction', 'semantics', 'machine_translation', 'knowledge_and_reasoning', 'summarization'])
dict_keys(['16_0', '17_0', '26_0', '26_1'])
dict_keys(['declarative', 'imperative', 'interrogative'])


In [26]:
# ANOVA WITH BONFERRONI
REJECT_THRESHOLD = 0.05

# 0 for TONE, 2 for GRAMMAR
CHAR_COLUMN = 0
task_lingcat_scores_by_model = {}
for mmm in all_tasktype_df_by_model.keys():
    task_lingcat_scores_by_model[mmm] = {}
    for tt in task_types:
        task_lingcat_scores_by_model[mmm][tt] = {}
        taskdf = all_tasktype_df_by_model[mmm][tt].drop(columns=['id'])
        cols = list(taskdf.columns)
        for i in range(0, len(cols), 4):
            chunk = cols[i:i+4]
            task_qidx = chunk[-1].split("-")[-2]
            if not any(task_qidx.startswith(prefix) for prefix in EXCLUDED_SCORES):
                task_lingcat_scores_by_model[mmm][tt][task_qidx] = {}
                chunkdf = taskdf[chunk].dropna()
                for ptone, tone_group in chunkdf.groupby(chunk[CHAR_COLUMN]):
                    task_lingcat_scores_by_model[mmm][tt][task_qidx][ptone] = tone_group[chunk[-1]].to_list()

for modl in task_lingcat_scores_by_model.keys():
    for tg in task_lingcat_scores_by_model[modl].keys():
        task_count_in_tg = len(task_lingcat_scores_by_model[modl][tg].keys())
        for qdi in task_lingcat_scores_by_model[modl][tg].keys():
            f, p = compute_anova_score(task_lingcat_scores_by_model[modl][tg][qdi])
            adjusted_p = p * task_count_in_tg * 7
            if (adjusted_p < REJECT_THRESHOLD):
                print(modl, "REJECTED!", f"task_group-{tg}", f"task-{qdi}", adjusted_p)

gpt_4o REJECTED! task_group-machine_translation task-2_3 5.4444766522276224e-05
gpt_4o REJECTED! task_group-machine_translation task-2_5 0.0011074897048645156
gpt_4o REJECTED! task_group-machine_translation task-2_2 8.862490932173876e-09
gpt_4o REJECTED! task_group-machine_translation task-2_1 0.000137164447936736
gpt_4o REJECTED! task_group-machine_translation task-2_6 2.564236080069202e-06
gpt_4o REJECTED! task_group-machine_translation task-2_8 1.7153883673422577e-09
gpt_4o REJECTED! task_group-machine_translation task-2_9 1.6767929786821897e-06
gpt_4o REJECTED! task_group-machine_translation task-2_7 0.0002774259635318474
llama31_8B_Inst REJECTED! task_group-machine_translation task-2_6 0.00013221625740463905
llama32_1B_Inst REJECTED! task_group-machine_translation task-2_11 0.035036966595370436
llama32_1B_Inst REJECTED! task_group-machine_translation task-2_5 0.038850884911009756
llama32_1B_Inst REJECTED! task_group-machine_translation task-2_7 0.0012878002935286772
llama32_1B_Ins

/Users/rayendito/miniconda3/envs/pietorch/lib/python3.12/site-packages/scipy/stats/_axis_nan_policy.py:573: ConstantInputWarning: Each of the input arrays is constant; the F statistic is not defined or infinite
  res = hypotest_fun_out(*samples, **kwds)
